# Bakehouse Sales Analysis

**Dataset:** `samples.bakehouse.sales_transactions`

**Difficulty:** Medium

**Topics:** date functions, window, running totals, ranking

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.read.table("samples.bakehouse.sales_transactions")

## Problem 1

Extract the date from `dateTime`. Compute daily total revenue and transaction count.
Sort results by `date` ascending.

**Expected output columns:**
- `date`
- `transaction_count`
- `total_revenue`

In [0]:
transactions.printSchema()

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = transactions.groupBy(
    F.to_date("dateTime").alias("date")
).agg(
    F.count("*").alias("transaction_count"),
    F.sum("totalPrice").alias("total_revenue")
).orderBy("date")

result_1.display()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'date' in cols, "Missing column: date"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rev = result_1.agg(F.min('total_revenue')).collect()[0][0]
assert min_rev >= 0, f"Expected total_revenue >= 0, found min={min_rev}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Calculate revenue and quantity sold per product. Rank products by revenue using a window function.

**Expected output columns:**
- `product`
- `total_revenue`
- `total_quantity`
- `revenue_rank`

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = transactions.groupBy("product").agg(
    F.sum("totalPrice").alias("total_revenue"),
    F.sum("quantity").alias("total_quantity")
).withColumn(
    "revenue_rank",
    F.rank().over(Window.orderBy(F.col("total_revenue").desc()))
)

result_2.display()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'product' in cols, "Missing column: product"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'total_quantity' in cols, "Missing column: total_quantity"
assert 'revenue_rank' in cols, "Missing column: revenue_rank"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rank = result_2.agg(F.min('revenue_rank')).collect()[0][0]
assert min_rank >= 1, f"Expected revenue_rank >= 1, found min={min_rank}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Compute the running total revenue ordered by `dateTime` using a window function.

**Expected output columns:**
- `transactionID`
- `dateTime`
- `totalPrice`
- `running_revenue`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = transactions.select(
    "transactionID",
    "dateTime",
    "totalPrice",
    F.sum("totalPrice").over(Window.orderBy("dateTime")).alias("running_revenue")
)

result_3.display()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
assert 'datetime' in cols, "Missing column: dateTime"
assert 'totalprice' in cols, "Missing column: totalPrice"
assert 'running_revenue' in cols, "Missing column: running_revenue"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_running = result_3.agg(F.max('running_revenue')).collect()[0][0]
assert max_running > 0, f"Expected running_revenue > 0, got max={max_running}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Find the top 3 products by revenue per payment method using window functions.
Keep only rows where rank <= 3.

**Expected output columns:**
- `paymentMethod`
- `product`
- `total_revenue`
- `rank`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = transactions.groupBy("paymentMethod", "product").agg(
    F.sum("totalPrice").alias("total_revenue")
).withColumn(
    "rank",
    F.rank().over(
        Window.partitionBy("paymentMethod") \
        .orderBy(F.col("total_revenue").desc())
    )
).filter(F.col("rank") <= 3)

result_4.display()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'paymentmethod' in cols, "Missing column: paymentMethod"
assert 'product' in cols, "Missing column: product"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'rank' in cols, "Missing column: rank"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rank = result_4.agg(F.max('rank')).collect()[0][0]
assert max_rank <= 3, f"Expected rank <= 3, found max rank={max_rank}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Calculate month-over-month revenue. Show year, month, revenue, and the previous month's revenue using a lag window function.

**Expected output columns:**
- `year`
- `month`
- `revenue`
- `prev_month_revenue`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5
w = Window.orderBy("month_first_date")
result_5 = transactions.groupBy(
    F.trunc("dateTime", "month").alias("month_first_date")
).agg(
    F.sum("totalPrice").alias("revenue")
).select(
    F.year("month_first_date").alias("year"),
    F.month("month_first_date").alias("month"),
    "revenue",
    F.lag("revenue").over(w).alias("prev_month_revenue")
)

result_5.display()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'year' in cols, "Missing column: year"
assert 'month' in cols, "Missing column: month"
assert 'revenue' in cols, "Missing column: revenue"
assert 'prev_month_revenue' in cols, "Missing column: prev_month_revenue"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Find all transactions where the quantity sold is above the average quantity for that product.
Use a window function to compute the per-product average.

**Expected output columns:**
- `transactionID`
- `product`
- `quantity`
- `avg_product_quantity`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6
w = Window.partitionBy("product").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
result_6 = transactions.select(
    "transactionID",
    "product",
    "quantity",
    F.avg("quantity").over(w).alias("avg_product_quantity")
).filter(F.col("quantity") > F.col("avg_product_quantity"))
result_6.display()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
assert 'product' in cols, "Missing column: product"
assert 'quantity' in cols, "Missing column: quantity"
assert 'avg_product_quantity' in cols, "Missing column: avg_product_quantity"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# Every quantity should be above avg
invalid = result_6.filter(F.col('quantity') <= F.col('avg_product_quantity')).count()
assert invalid == 0, f"Found {invalid} rows where quantity <= avg_product_quantity"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Calculate the percentage share of each product's revenue out of total revenue.
Sort by `revenue_pct` descending.

**Expected output columns:**
- `product`
- `total_revenue`
- `revenue_pct`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
result_7 = transactions.groupBy("product").agg(
    F.sum("totalPrice").alias("total_revenue")
).withColumn(
    "revenue_pct",
    100*F.col("total_revenue")/F.sum("total_revenue").over(w)
).orderBy(F.col("revenue_pct").desc())

result_7.display()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'product' in cols, "Missing column: product"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'revenue_pct' in cols, "Missing column: revenue_pct"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_pct = result_7.agg(F.max('revenue_pct')).collect()[0][0]
min_pct = result_7.agg(F.min('revenue_pct')).collect()[0][0]
assert max_pct <= 100, f"Expected revenue_pct <= 100, got max={max_pct}"
assert min_pct >= 0, f"Expected revenue_pct >= 0, got min={min_pct}"
print(f"Problem 7 passed ✓  ({cnt} rows)")